# Notebook 3: Multimodal Travel Guide

![Workshop flow — you are here: Tool Calling & Multimodal AI](../assets/notebook_flow_diagram.png)

Imagine you are travelling from your city to an African city for a workshop. You want:

- real weather and exchange-rate data (from tools, not guesses)
- a written briefing from the LLM
- optional audio and a simple poster

This is our **multimodal** session: text, tools, audio, and image in one small project.


## Step 1: Setup


In [1]:
import sys
from pathlib import Path

# Run this cell first. It finds the project folder and loads your .env file.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")


False

In [2]:
from src.llm_gateway import check_available_providers, run_llm

SELECTED_PROVIDER = "auto"
check_available_providers()

{'openai': False,
 'anthropic': False,
 'gemini': False,
 'mistral': False,
 'cohere': False,
 'deepseek': False,
 'groq': False,
 'openrouter': False,
 'ollama': True}

## Step 2: Choose your trip

Edit the values if you like. The defaults are Kigali → Accra.


In [3]:
base_city = "Kigali"
base_country = "Rwanda"
destination_city = "Accra"
destination_country = "Ghana"
base_currency = "RWF"
destination_currency = "GHS"
traveller_profile = "data scientist attending an AI workshop" 


## Step 3: Call each tool separately

**Rule:** the LLM should not invent weather or exchange rates. We fetch them first.


In [ ]:
import pandas as pd
from src.travel_tools import gather_travel_tool_results
from src.output_formatting import format_tool_evidence_dataframe, format_places_dataframe

evidence = gather_travel_tool_results(
    base_city, base_country,
    destination_city, destination_country,
    base_currency, destination_currency,
    amount=100,
)

format_tool_evidence_dataframe(evidence)


In [ ]:
format_places_dataframe(evidence["places"])


## Step 4: Grounded prompt + LLM answer

Now we pass the tool results into the prompt.


In [ ]:
from IPython.display import Markdown, display
from src.prompt_templates import build_travel_brief_prompt
from src.output_formatting import format_travel_brief_markdown

prompt = build_travel_brief_prompt(
    base_city, base_country,
    destination_city, destination_country,
    traveller_profile,
    evidence["weather"],
    evidence["distance"],
    evidence["exchange"],
    evidence["places"],
)

travel_reply = run_llm(prompt, provider=SELECTED_PROVIDER)
display(
    Markdown(
        format_travel_brief_markdown(
            travel_reply,
            base_city,
            base_country,
            destination_city,
            destination_country,
        )
    )
)


## Step 5: Audio (optional)

On macOS this works even without extra packages (built-in `say`). Otherwise install `edge-tts` or `gTTS`.


In [ ]:
from IPython.display import Audio, display
from src.image_generation import text_to_speech
from src.output_formatting import strip_markdown_for_speech

audio_path = text_to_speech(strip_markdown_for_speech(travel_reply))
display(Audio(filename=audio_path))
audio_path


## Step 6: Travel poster

Uses an image API if you have a key; otherwise builds a simple poster with Pillow.


In [ ]:
from IPython.display import Image, display
from src.image_generation import generate_travel_poster

poster = generate_travel_poster(destination_city, destination_country, evidence["places"])
display(Image(filename=poster["path"]))
poster


## Step 7: Full Gradio app

This combines tools, the LLM brief, audio, and poster in one interface.
Watch the **Status** box — generation can take 20–40 seconds.


In [ ]:
from src.gradio_apps import build_tour_guide_app

tour = build_tour_guide_app()
# Watch the Status box while tools, LLM, audio, and poster run.
tour.launch()


## Reflection

1. What should the model **not** guess?
2. Why call external tools first?
3. What makes this app multimodal?
4. What could go wrong if weather or rates were hallucinated?
5. How would you adapt this for a university conference assistant?

**Verify before travel**: this is a teaching demo, not live travel advice.
